# 02 — Build resized study cache

Attach competition data + `rsna-knee-code`.

1. `LIMIT = 50` smoke test  
2. `LIMIT = 0` full run  
3. Save Version → New Dataset `rsna-knee-cache-v1`

In [ ]:
from pathlib import Path
import sys

REPO = Path('/kaggle/input/datasets/girishbose/rsna-knee-code')
DATA = Path('/kaggle/input/competitions/rsna-knee-abnormality-detection')

if not (REPO / 'src' / 'rsna_knee').exists():
    hits = [h for h in REPO.rglob('rsna_knee') if h.is_dir() and h.parent.name == 'src']
    if not hits:
        raise SystemExit('src/rsna_knee not found under code dataset')
    REPO = hits[0].parent.parent

sys.path.insert(0, str(REPO / 'src'))

LIMIT = 50  # set to 0 for full 4407 studies
OUT = Path('/kaggle/working/cache_v1')
OUT.mkdir(parents=True, exist_ok=True)

print('REPO', REPO)
print('DATA', DATA)
print('build_cache.py?', (REPO / 'scripts' / 'build_cache.py').exists())
print('train.csv?', (DATA / 'train.csv').exists())
print('train_series/?', (DATA / 'train_series').exists())

In [ ]:
import os, subprocess

%pip -q install pydicom opencv-python-headless tqdm pyyaml

env = os.environ.copy()
env['PYTHONPATH'] = str(REPO / 'src') + (os.pathsep + env['PYTHONPATH'] if env.get('PYTHONPATH') else '')

cmd = [
    sys.executable, str(REPO / 'scripts' / 'build_cache.py'),
    '--train-csv', str(DATA / 'train.csv'),
    '--series-csv', str(DATA / 'train_series.csv'),
    '--series-root', str(DATA / 'train_series'),
    '--out-dir', str(OUT),
    '--max-series', '3',
    '--n-slices', '12',
    '--image-size', '224',
]
if LIMIT:
    cmd += ['--limit', str(LIMIT)]

print('PYTHONPATH=', env['PYTHONPATH'])
print(' '.join(cmd))
subprocess.check_call(cmd, env=env)
print('cache files', len(list(OUT.glob('*.npz'))))